In [1]:
from xact.log.config import log_manager
from xact.config.gen import config

log_manager.config(level=20)


from xact.llm import llm_client


print(config.XACT_LLM_MODEL)

from xact.llm.llm import LLM


# log.enable(True)
# config.XACT_LLM_MODEL = "qwen2.5:1.5b "

log = log_manager.init(__name__)

llm = LLM(model="llama3.2:3b")

Feb/12 02:19:08 |   xact.log.config     | INFO     | log initilized
Feb/12 02:19:08 |   xact.log.config     | XACT_STREAM | log streaming initilized
Feb/12 02:19:08 |   xact.config.gen     | INFO     | Config loaded from .env:.env json_path : config.json


qwen2.5:1.5b


In [2]:
llm.list()

['deepseek-r1:7b',
 'deepseek-r1:1.5b',
 'deepseek-coder:1.3b',
 'granite3.1-moe:latest',
 'llama3.2:3b',
 'codegemma:2b',
 'granite3.1-dense:2b',
 'gemma2:2b',
 'llava:latest',
 'llama3.2:1b',
 'nomic-embed-text:latest',
 'mistral:latest',
 'nemotron-mini:latest',
 'qwen2.5-coder:1.5b',
 'qwen2.5:3b',
 'qwen2.5:1.5b']

In [3]:
log_manager.loggers

{'xact.log.config': <Logger xact.log.config (INFO)>,
 'xact-stream': <Logger xact-stream (INFO)>,
 'xact': <Logger xact (INFO)>,
 'xact.config.gen': <Logger xact.config.gen (INFO)>,
 'xact.llm.llm': <Logger xact.llm.llm (DEBUG)>,
 '__main__': <Logger __main__ (DEBUG)>}

In [4]:
config.XACT_LLM_MODEL = "qwen2.5:1.5b"

In [5]:
log_manager.ext_enable()
log_manager.enable()

In [6]:
llmout = llm.generate(prompt="hi", gen_format="gen")

Feb/12 02:19:19 |     xact.llm.llm      | INFO     | llm out generated


In [7]:
llmout.data

'How can I assist you today?'

In [8]:
def get_data(year: int, month: int, day: int):
    """return data for given date"""
    return "data"


from xact.data.data import DataX


from xact.flow.tool.tool import gen_function_schema, tool
from xact.utils.gen import gen_datetime, gen_uuid

from xact.flow.router.router import RouteData
from xact.flow.router.router import Router

tools = [
    get_data,
    gen_uuid,
    tool(func=gen_datetime),
]
routedata = RouteData()
routedata.embed(data_list=tools + [DataX(content="data")])
routedata.data_embd_str

Feb/12 02:19:26 |    xact.llm.embed     | INFO     | llm embed generated


[' get_data description : return data for given date function : [{"type": "function", "function": {"name": "get_data", "description": "return data for given date", "parameters": {"type": "object", "properties": {"year": {"type": "integer"}, "month": {"type": "integer"}, "day": {"type": "integer"}}, "required": ["year", "month", "day"]}}}]',
 ' gen_uuid description : generate uuid function : [{"type": "function", "function": {"name": "gen_uuid", "description": "generate uuid", "parameters": {"type": "object", "properties": {}, "required": []}}}]',
 'gen_datetime description : give local_utc or utc time function : [{"type": "function", "function": {"name": "gen_datetime", "description": "give local_utc or utc time", "parameters": {"type": "object", "properties": {"local_utc": {"type": "boolean"}}, "required": []}}}]',
 'data']

In [9]:
res = Router.route_vector(prompt="give some unique id", route_data=routedata)
res

Feb/12 02:19:26 |    xact.llm.embed     | INFO     | llm embed generated


SearchMDResponse(idx=[1, 3, 2, 0], score=[0.45732447826959066, 0.45180458417474734, 0.42409111301663527, 0.41729361859076897], data=[<function gen_uuid at 0x7f0735974360>, DataX(uid='eb66ccb4-215d-4ce5-8f87-cd18d9915192', cid=None, flow_mode='prompt', role='xact', content='data', content_type='str', md_content=None, metadata=None, tags=None, description=None, source=None, embed=None, embed_id=None, created_at=datetime.datetime(2025, 2, 12, 2, 19, 19, 686373), time_triggers=None), <xact.flow.tool.tool.Tool object at 0x7f0734c79fd0>, <function get_data at 0x7f073e5dac00>], data_embed=[])

In [10]:
llm.model.var = "llama3.2:1b"
messages = [
    {
        "role": "system",
        "content": f"you are ox-ai helpful ai assistant you are excelent at resonaing and assisting in any tasks you are given with list of tools u need to pick whihc tool is best suited for the task toos : {routedata.data_embd_str}   give the tool name as output in json format",
    },
    {
        "role": "user",
        "content": "task : to find an unique id \n\n give the tool name that can performa the task tool_name : 'name of the tool' ",
    },
]
from pydantic import BaseModel


class ToolResponse(BaseModel):
    toolname: str


res = llm.generate(messages=messages, response_format=ToolResponse)

print(res)

Feb/12 02:19:35 |     xact.llm.llm      | INFO     | llm out generated


data=ToolResponse(toolname='gen_uuid') completion=ParsedChatCompletion[ToolResponse](id='chatcmpl-580', choices=[ParsedChoice[ToolResponse](finish_reason='stop', index=0, logprobs=None, message=ParsedChatCompletionMessage[ToolResponse](content='{\n    "toolname": "gen_uuid"\n}', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=[], parsed=ToolResponse(toolname='gen_uuid')))], created=1739306975, model='llama3.2:1b', object='chat.completion', service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=12, prompt_tokens=326, total_tokens=338, completion_tokens_details=None, prompt_tokens_details=None)) gen_format='struct'


In [11]:
res.data, llm.model

(ToolResponse(toolname='gen_uuid'), llama3.2:1b)

In [12]:
def my_function(param1: str, param2: int, param3: bool = True, **kwargs) -> str:
    """
    This function does something.

    Args:
        param1: A string parameter.  (Description of param1)
        param2: An integer parameter. (Description of param2)
        param3: An optional boolean parameter. Defaults to True. (Description of param3)

    Returns:
        A string.
    """
    # ... function code ...
    print(kwargs)
    return "result"

In [13]:
mt = tool(func=my_function)
mt.get_schema()

{'type': 'function',
 'function': {'name': 'my_function',
  'description': 'This function does something.\n\n    Args:\n        param1: A string parameter.  (Description of param1)\n        param2: An integer parameter. (Description of param2)\n        param3: An optional boolean parameter. Defaults to True. (Description of param3)\n\n    Returns:\n        A string.',
  'parameters': {'type': 'object',
   'properties': {'param1': {'type': 'string'},
    'param2': {'type': 'integer'},
    'param3': {'type': 'boolean'},
    'kwargs': {'type': 'string'}},
   'required': ['param1', 'param2', 'kwargs']}}}

In [14]:
class Flow:
    def __init__(
        self,
        mode: str,
    ):
        self.mode = mode

In [15]:
# Example Usage:

from xact.flow.act.act import Action


def shutdown():
    return "shutting down"


shutdown_action = Action(
    func=shutdown
)  # No name provided, should auto-assign "shutdown"


def open_apps(name: str):
    """open apps from the given name"""
    return f"{name} app opened"


open_apps_action = Action(
    name="open_apps", description="open apps from the given name", func=open_apps
)

print(open_apps_action.func(name="chrome"))  # Output: "chrome app opened"

def get_data(prompt:str,date:str):
    "serach and return data from db using all search keywoards"
    return prompt,date

get_data_act = Action(func=get_data)


print(Action.get_actions())

Feb/12 02:19:42 |    xact.llm.embed     | INFO     | llm embed generated
Feb/12 02:19:42 |    xact.llm.embed     | INFO     | llm embed generated
Feb/12 02:19:42 |    xact.llm.embed     | INFO     | llm embed generated


chrome app opened
{'shutdown': <xact.flow.act.act.Action object at 0x7f073595a9d0>, 'open_apps': <xact.flow.act.act.Action object at 0x7f0734bc0650>, 'get_data': <xact.flow.act.act.Action object at 0x7f0734b257d0>}


In [16]:
shutdown_action.act(prompt="shutdown the system")

Feb/12 02:19:42 |  xact.flow.act.act    | INFO     | executing action : shutdown
Feb/12 02:19:42 | xact.flow.tool.tool   | INFO     | executing tool : shutdown


'shutting down'

In [17]:
var = str(Action.llm.model)

print(var)

qwen2.5:1.5b


In [18]:
from xact.flow.tool.tool import Tool
from xact.flow.router.router import RouteData

rout_tools = RouteData()
rout_tools.embed([Tool(open_apps)])
Action.nact(prompt="open youtube", route_tools=rout_tools, kwargs={"name": "chrome"})

Feb/12 02:19:43 |    xact.llm.embed     | INFO     | llm embed generated
Feb/12 02:19:43 |    xact.llm.embed     | INFO     | llm embed generated
Feb/12 02:19:52 |     xact.llm.llm      | INFO     | llm out generated
Feb/12 02:19:52 | xact.flow.tool.tool   | INFO     | executing tool : open_apps


'chrome app opened'

In [19]:
open_apps_action.act(prompt="open youtube", kwargs={"name": "chrome"})

Feb/12 02:19:52 |  xact.flow.act.act    | INFO     | executing action : open_apps
Feb/12 02:19:52 | xact.flow.tool.tool   | INFO     | executing tool : open_apps


'chrome app opened'

In [20]:
get_data_act.tool.fun_schema

{'type': 'function',
 'function': {'name': 'get_data',
  'description': 'serach and return data from db using all search keywoards',
  'parameters': {'type': 'object',
   'properties': {'prompt': {'type': 'string'}, 'date': {'type': 'string'}},
   'required': ['prompt', 'date']}}}

In [21]:
Action.nact(prompt="get data 2/07/2024",kwargs={
    "prompt":"get latest data my llm notes and xact",
})

Feb/12 02:19:58 |    xact.llm.embed     | INFO     | llm embed generated
Feb/12 02:20:08 |     xact.llm.llm      | INFO     | llm out generated
Feb/12 02:20:08 | xact.flow.tool.tool   | INFO     | executing tool : get_data


('get latest data my llm notes and xact', '2/07/2024')

In [34]:
import inspect
from typing import List


def get_data_(prompt:List[int],date:str):
    "serach and return data from db using all search keywoards"
    return prompt,date

func =get_data_
type_map = {
    str: "string",
    int: "integer",
    float: "number",
    bool: "boolean",
    list: "array",
    dict: "object",
    type(None): "null",
}

try:
    signature = inspect.signature(func)
except ValueError as e:
    raise ValueError(
        f"Failed to get signature for function {func.__name__}: {str(e)}"
    )

parameters = {}
for param in signature.parameters.values():
    try:
        print(param.annotation)
        param_type = type_map.get(param.annotation, "string")
    except KeyError as e:
        raise KeyError(
            f"Unknown type annotation {param.annotation} for parameter {param.name}: {str(e)}"
        )
    parameters[param.name] = {"type": param_type}


typing.List[int]
<class 'str'>


In [22]:
from xact.flow.tool.tool import tool


@tool(
    description="mf",
    param_description={
        "param1": "A string parameter.",
        "param2": "An integer parameter.",
        "param3": "An optional boolean parameter. Defaults to True.",
    },
)
def my_f(
    hhh: str, param2: int, param3: bool = True
) -> str:  # Type hints for parameters and return value
    """
    This function does something.

    Args:
        param1: A string parameter.  (Description of param1)
        param2: An integer parameter. (Description of param2)
        param3: An optional boolean parameter. Defaults to True. (Description of param3)

    Returns:
        A string.
    """
    # ... function code ...
    return "result"


def my_ff(
    hhh: str, param2: int, param3: bool = True
) -> str:  # Type hints for parameters and return value
    """
    This function does something.

    Args:
        param1: A string parameter.  (Description of param1)
        param2: An integer parameter. (Description of param2)
        param3: An optional boolean parameter. Defaults to True. (Description of param3)

    Returns:
        A string.
    """
    # ... function code ...
    return "result"

In [23]:
print(my_f(1, 2))

Feb/12 02:20:08 | xact.flow.tool.tool   | INFO     | executing tool : my_f


result


In [24]:
print(tool(func=my_f)(1, 2))

Feb/12 02:20:09 | xact.flow.tool.tool   | INFO     | executing tool : my_f


result


In [25]:
from datetime import datetime


def get_weather(location: str):
    print(location)
    return location


tools = [
    gen_function_schema(get_data),
    gen_function_schema(gen_uuid),
    gen_function_schema(gen_datetime),
    Tool(
        get_weather,
        description="just give one place name as input",
        param_description={"location": "location name of one place only"},
    ).get_schema(),
]

model = "qwen2.5:1.5b"
model = "llama3.2:3b"
# model = "granite3.1-dense:2b"
completion = llm_client.chat.completions.create(
    model=model,
    messages=[
        {"role": "user", "content": "What is the weather in mumbai bangaore chennai"}
    ],
    tools=tools,
    tool_choice="required",
    temperature=0,
)

print(completion.choices[0].message.tool_calls)
for cmp in completion.choices[0]:
    print(cmp)  # .choices[0].message.tool_calls)
    print("------")

[ChatCompletionMessageToolCall(id='call_cbiswigh', function=Function(arguments='{"location":"Mumbai, Bangalore, Chennai"}', name='get_weather'), type='function', index=0)]
('finish_reason', 'tool_calls')
------
('index', 0)
------
('logprobs', None)
------
('message', ChatCompletionMessage(content='', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_cbiswigh', function=Function(arguments='{"location":"Mumbai, Bangalore, Chennai"}', name='get_weather'), type='function', index=0)]))
------


In [26]:
print(completion)
for cmp in completion:
    print(cmp)  # .choices[0].message.tool_calls)
    print("------")

ChatCompletion(id='chatcmpl-141', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_cbiswigh', function=Function(arguments='{"location":"Mumbai, Bangalore, Chennai"}', name='get_weather'), type='function', index=0)]))], created=1739307020, model='llama3.2:3b', object='chat.completion', service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=18, prompt_tokens=310, total_tokens=328, completion_tokens_details=None, prompt_tokens_details=None))
('id', 'chatcmpl-141')
------
('choices', [Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_cbiswigh', function=Function(arguments='{"location":"Mumbai, Bangalore, Chennai"}', na

In [27]:
from datetime import date, timedelta

# Get the current date
today = date.today()
print("Today:", today)

# Add one day
tomorrow = today + timedelta(days=1)
print("Tomorrow:", tomorrow)

# Add multiple days
in_three_days = today + timedelta(days=3)
print("In three days:", in_three_days)

# Subtract days (go to the past)
yesterday = today - timedelta(days=1)
print("Yesterday:", yesterday)

# Working with specific dates:
specific_date = date(2024, 1, 1)  # January 1, 2024
next_day = specific_date + timedelta(days=1)
print(f"The day after {specific_date}: {next_day}")

# Example with months/years:
# Note: timedelta only works with days, seconds, microseconds, milliseconds, minutes, hours, and weeks.
# For month or year arithmetic, you need a different approach (see below).

Today: 2025-02-12
Tomorrow: 2025-02-13
In three days: 2025-02-15
Yesterday: 2025-02-11
The day after 2024-01-01: 2024-01-02


In [28]:
from xact.config.gen import Config

In [29]:
print(res)

data=ToolResponse(toolname='gen_uuid') completion=ParsedChatCompletion[ToolResponse](id='chatcmpl-580', choices=[ParsedChoice[ToolResponse](finish_reason='stop', index=0, logprobs=None, message=ParsedChatCompletionMessage[ToolResponse](content='{\n    "toolname": "gen_uuid"\n}', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=[], parsed=ToolResponse(toolname='gen_uuid')))], created=1739306975, model='llama3.2:1b', object='chat.completion', service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=12, prompt_tokens=326, total_tokens=338, completion_tokens_details=None, prompt_tokens_details=None)) gen_format='struct'


In [30]:
config.AUDIO_FORMAT

8